**The correct way to use `lru_cache`**

In [1]:
from functools import lru_cache

@lru_cache(maxsize=None)   # unlimited cache, same as your memoize
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

- automatic memoization
- thread‑safe caching
- introspection tools
- cache statistics
- cache clearing
- LRU eviction (if you set a maxsize)

**What lru_cache actually does**:

`lru_cache` wraps your function in a **C‑optimized** caching layer:
- Keys are built from the **function arguments**
- Values are stored in an ordered dictionary
- If `maxsize` is set, the least‑recently‑used entry is evicted
- If `maxsize=None`, the cache grows without limit (pure memoization)
- The wrapper preserves metadata via `functools.wraps`
- The wrapper is thread‑safe

Your custom `memoize` decorator is essentially a simplified version of this.

**How to inspect the cache**
```py
fibonacci.cache_info()
```
Output:
```shell
CacheInfo(hits=98, misses=10, maxsize=None, currsize=10)
```
Clear the cache:
```py
fibonacci.cache_clear()
```

---

**How to limit memory usage (true LRU mode)**
```py
@lru_cache(maxsize=256)
def fibonacci(n):
    ...
```
Now the cache holds only the 256 **most recently used** results.

---

**How to use `lru_cache` with methods (important!)**

If you decorate a method, the self argument becomes part of the cache key.
To avoid that, use `@staticmethod` or move the cached function outside the class.

---

### What LRU eviction actually means
```py
@lru_cache(maxsize=256)
```
you are enabling **Least Recently Used** eviction.

**LRU ≠ FIFO**
- FIFO = First In, First Out  
- LRU = Least Recently Used

LRU tracks **recency of access**, not insertion order.

**Example**

Cache size = 3

Access pattern:
```shell
A → B → C → A → D
```
**Eviction happens when inserting `D`**.

Which key is evicted?
- FIFO would evict A (first inserted)
- LRU evicts B (least recently used)

Because the access order was:
```shell
A (used recently)
C (used recently)
B (oldest access)
```
So LRU removes `B`.

**Python’s lru_cache uses a doubly‑linked list + dict**

This gives:
- $O(1)$ lookup
- $O(1)$ update of recency
- $O(1)$ eviction

Your custom memoize uses only a dict → no eviction logic.

---

### What “thread‑safe” means in `lru_cache`
`lru_cache` uses an internal **reentrant lock** (RLock) around all cache operations:
- reading from the cache
- writing to the cache
- updating the LRU order
- evicting items

This ensures that only one thread at a time mutates the internal structures.

---

**`lru_cache` wraps all cache operations in*:
```py
with self.lock:
    ...
```
- no concurrent writes
- no corrupted state
- consistent LRU ordering
- safe eviction
- safe recursion
- safe multi‑threaded access

---

**Why this matters**

Imagine two threads calling the same memoized function at the same time:
```shell
Thread A: checks if args in cache
Thread B: checks if args in cache
Thread A: inserts into cache
Thread B: inserts into cache
```
Without locking, this can cause:
- corrupted dictionary state
- lost writes
- inconsistent cache entries
- race conditions
- `KeyError` or `ValueError` from dict resizing

Python dicts are not thread‑safe for concurrent writes.